# 机器学习模块入门

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 1 / 34 步：建立训练、切分与预处理工作流**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 模块入门与应用任务  →  **本章任务：** 机器学习模块入门  →  **下一步：** Scikit-learn工作流与数据切分
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景



到此之前，我们都在"描述"数据：清洗、汇总、画图。从本模块起，我们开始做更有难度的事——**预测**（给新数据一个结果，如"这个客户会不会流失"）和**分群**（把相似样本归到一类）。



**Scikit-learn（sklearn）** 是 Python 机器学习的标准库。它的 API 高度统一：`构造模型 → fit 训练 → predict 预测 → score 评估`。四种经典场景（分类、回归、聚类、降维）都能用它完成。学完这个模块，你就能独立完成一个"从数据到预测"的完整小项目。


## 本章目标

学完本章，你将能够：

- **理解**：理解机器学习解决什么问题、有哪些任务类型与整体工作流（数据→特征→模型→评估）。
- **操作**：能完成一次最简单的监督学习训练并读出评估指标。
- **迁移**：能判断一个业务问题是否适合用机器学习，并搭出第一步流水线。


## 86.1 这个模块解决什么问题



**背景引入**：老板想提前知道"哪些客户会流失"，好做优惠挽留；想知道"哪些地区下月销量会涨"。这些都不能直接翻表回答，需要从历史数据里"学"规律。（打个比方：机器学习不像“背公式”，而像看多之后自然学会“看云识天气”——你见过几百次下雨前的天空，就能凭哪种云预测下雨，而不必记住每朵云的公式。）



- **统一 API**：fit / predict / score 一套流程通吃所有模型；

- **四大任务**：分类、回归、聚类、降维；

- **工程化**：数据切分、特征工程、模型评估与调优一条龙。


## 86.2 学习地图



本模块遵循一条完整建模工作流：



1. **80.3** 理解工作流，切分训练集 / 测试集（防数据泄漏）；

2. **80.4 数据预处理 / Pipeline**：标准化、缺失、管道化；

3. **80.5-80.10 算法基座**：线性/逻辑回归、KNN、决策树、随机森林、梯度提升、SVM、朴素贝叶斯；

4. **80.11 无监督**：K-Means 聚类、PCA 降维；

5. **80.12-80.13 调优**：交叉验证、超参数搜索；

6. **80.14-80.16 评估进阶**：不平衡分类、多分类、混淆矩阵、ROC、概率校准；

7. **80.17+ 项目**：用户消费、物流延期、共享单车、银行营销四大综合项目。



**重点**：机器学习质量 **80% 取决于数据质量**，前面 Pandas 的清洗与特征构建正是本模块的基石。


## 86.3 第一个上手示例



感受"训练→预测"的最小闭环。运行下方代码：用鸢尾花数据训练一个 KNN 分类器，预测一朵花的品种。


<!-- math-foundation:intro-machine-learning -->
### 数学推导｜机器学习是在未见数据上最小化风险

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先定义单个样本的损失。** 预测函数 $f$ 在样本 $(x_i,y_i)$ 上产生 $L_i=L(y_i,f(x_i))$。

**第 2 步｜真正关心的是未来总体风险。** 若未来数据来自分布 $P$，理想目标是

$$
R(f)=\mathbb{E}_{(X,Y)\sim P}[L(Y,f(X))]
$$

**第 3 步｜用训练样本近似未知期望。** 经验风险 $\hat R(f)=\sum_iL_i/n$ 是可计算代理。模型在训练集最小化它，验证集估计方案选择后的泛化表现，测试集只做最终审计。

**把上面的关系收束为本章计算式：**

$$
\hat{R}(f)=\frac{1}{n}\sum_{i=1}^{n}L\bigl(y_i,f(x_i)\bigr)
$$

**符号解释：** $L$ 是损失函数，$\hat{R}$ 是样本上的经验风险。

**代码对应：** 训练只使用训练集拟合；验证集选方案；测试集只做最终一次评估。

**使用边界：** 训练误差低不代表泛化好；数据泄漏会让评估虚高。


In [ ]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)

model = KNeighborsClassifier(n_neighbors=3)  # 构造模型
model.fit(X_train, y_train)  # 训练
acc = model.score(X_test, y_test)  # 评估
print("测试集准确率: %.2f" % acc)


<!-- module-intro-checkpoint:intro-machine-learning -->
## 86.4 入门验收｜先建立不泄漏的基线评估

**应用背景：** 在尝试复杂模型前，先切分数据并建立一个最简单的分类基线，确认后续模型究竟有没有带来真实改进。

**你需要完成：**

1. 按 75% / 25% 切分鸢尾花数据，并使用 `stratify=y`。
2. 只在训练集上拟合 `DummyClassifier`。
3. 输出测试集准确率，作为后续模型比较基线。

**操作提示：** 固定 `random_state=42`；先切分，再调用 `fit(X_train, y_train)`。

**验收标准：** 训练与测试样本分开，基线只拟合训练集，并成功输出 0 到 1 之间的准确率。


In [ ]:
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)

# TODO 1：完成分层切分
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=...,
    random_state=...,
    stratify=...,
)

# TODO 2：拟合多数类基线并输出测试准确率
baseline = DummyClassifier(strategy="most_frequent")
...
baseline_accuracy = ...
print("baseline accuracy:", baseline_accuracy)


In [ ]:
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_accuracy = baseline.score(X_test, y_test)
print("baseline accuracy:", baseline_accuracy)


## 86.5 小结
### 你已经了解



- 机器学习做"预测 + 分群"，用 sklearn 的标准流程 fit / predict / score；

- 必须先把数据切分为训练集 / 测试集，防止"学习测试结果"；

- 模型质量大前提是数据质量，前面 Pandas 是基石。



### 自测清单



- [ ] 能说出 sklearn 统一 API 的四步（构造/fit/predict/score）；

- [ ] 能解释为什么要切分训练集与测试集；

- [ ] 能运行一个最小分类模型并读准确率。



> **模块核心口诀**：训练集学规律，测试集验真假；先基线后模型，先数据后调参。
